In [7]:
import os
import zipfile
import shutil
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd

In [20]:
ZIP_FILES = ['train.zip', 'valid.zip', 'test.zip']

# 解压数据集 - 每个zip解压到对应的目录
def extract_zips(zip_files):
    """解压所有zip文件到对应的目录"""
    for zip_file in zip_files:
        if os.path.exists(zip_file):
            # 获取目录名（去掉.zip后缀）
            dir_name = zip_file.replace('.zip', '')
            # 创建目录（如果不存在）
            os.makedirs(dir_name, exist_ok=True)
            print(f"正在解压 {zip_file} 到 {dir_name}/...")
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(dir_name)
            print(f"{zip_file} 解压完成")
        else:
            print(f"警告: {zip_file} 不存在")

# 解压数据（如果还没解压）
if not os.path.exists('train'):
    extract_zips(ZIP_FILES)
else:
    print("数据已经解压，跳过解压步骤")

# 检查解压后的文件结构
print("\n解压后的文件结构：")
for dir_name in ['train', 'valid', 'test']:
    if os.path.exists(dir_name):
        subdirs = [d for d in os.listdir(dir_name) if os.path.isdir(os.path.join(dir_name, d))]
        print(f"{dir_name}/: {subdirs}")
        for subdir in subdirs:
            subdir_path = os.path.join(dir_name, subdir)
            file_count = len([f for f in os.listdir(subdir_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
            print(f"  - {subdir}/: {file_count} 张图片")
    else:
        print(f"警告: {dir_name}/ 目录不存在，请先解压数据")


正在解压 train.zip 到 train/...
train.zip 解压完成
正在解压 valid.zip 到 valid/...
valid.zip 解压完成
正在解压 test.zip 到 test/...
test.zip 解压完成

解压后的文件结构：
train/: ['nowildfire', 'wildfire']
  - nowildfire/: 14500 张图片
  - wildfire/: 15750 张图片
valid/: ['nowildfire', 'wildfire']
  - nowildfire/: 2820 张图片
  - wildfire/: 3480 张图片
test/: ['nowildfire', 'wildfire']
  - nowildfire/: 2820 张图片
  - wildfire/: 3480 张图片


In [11]:
class WildfireDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        """
        Args:
            data_dir: Path to folder containing 'wildfire' and 'nowildfire' subfolders
            transform: Optional transform to apply to images
        """
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Class mapping
        class_map = {'wildfire': 1, 'nowildfire': 0}

        # Load image paths and labels
        # both wildfire and nowildfire image paths go into one array
        for class_name, label in class_map.items():
            class_path = os.path.join(data_dir, class_name)

            if os.path.exists(class_path):
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith(('.jpg', 'jpeg')):
                        self.image_paths.append(os.path.join(class_path, img_name))
                        self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    # pulls actual image from class attribute image paths
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB') # just in case there are grayscale images
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [56]:
# 定义数据路径（数据解压到当前目录）
EXTRACT_BASE_PATH = '.'

# 注意：数据集将在后面添加transform后创建（见Cell 8）

print("数据路径已定义，将在添加transform后创建数据集")

数据路径已定义，将在添加transform后创建数据集


In [57]:
train_dataset.image_paths[0:5]

['./train/wildfire/-75.79731,47.6256.jpg',
 './train/wildfire/-69.5572,51.9064.jpg',
 './train/wildfire/-79.11752,47.38459.jpg',
 './train/wildfire/-72.743,45.7617.jpg',
 './train/wildfire/-75.7764,45.4516.jpg']

In [58]:
# 注意：数据加载器将在Cell 8中创建（在添加transform之后）
# 这里先跳过，避免使用没有transform的数据
print("数据加载器将在添加transform后创建")

数据加载器将在添加transform后创建


In [27]:
type(train_dataloader)

torch.utils.data.dataloader.DataLoader

In [49]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

model = models.resnet18(weights='IMAGENET1K_V1')  # 使用新的API

# 修改最后一层以适应二分类任务（wildfire/nowildfire）
num_classes = 2
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

print(f"模型结构已修改：最后一层输出维度 = {num_classes}")

# 检查是否有GPU可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Facility Equipment {device}")
model = model.to(device)


模型结构已修改：最后一层输出维度 = 2
Facility Equipment cpu


In [50]:
# ==================== 4. 定义损失函数和优化器 ====================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 可选：学习率调度器
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [51]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)
        
        # 前向传播
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        # 统计
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

# ==================== 6. 验证函数 ====================
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc




In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
import time

In [ ]:

# ==================== 7. 主训练循环 ====================
num_epochs = 20
best_val_acc = 0.0

print('开始训练...\n')
for epoch in range(num_epochs):
    start_time = time.time()
    
    # 训练
    train_loss, train_acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)
    
    # 验证
    val_loss, val_acc = validate(model, val_dataloader, criterion, device)
    
    # 学习率调度
    scheduler.step()
    
    # 打印信息
    epoch_time = time.time() - start_time
    print(f'Epoch [{epoch+1}/{num_epochs}] ({epoch_time:.2f}s)')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print()
    
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  *** 保存最佳模型 (Val Acc: {val_acc:.2f}%) ***\n')

print('训练完成！')

开始训练...



KeyboardInterrupt: 

In [ ]:
# 添加数据预处理transform（如果数据集还没有transform）
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])  # ImageNet标准化
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

# 重新创建带transform的数据集
train_dataset = WildfireDataset(os.path.join(EXTRACT_BASE_PATH, 'train'), transform=train_transform)
val_dataset = WildfireDataset(os.path.join(EXTRACT_BASE_PATH, 'valid'), transform=val_test_transform)
test_dataset = WildfireDataset(os.path.join(EXTRACT_BASE_PATH, 'test'), transform=val_test_transform)

# 重新创建DataLoader（验证集和测试集不打乱）
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)



数据加载器已创建


In [34]:
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)  # 每7个epoch降低学习率

In [37]:
# 训练函数
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)
        
        # 前向传播
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        # 统计
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

# 验证函数
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

print("训练和验证函数已定义")

训练和验证函数已定义


In [38]:
# 训练模型
num_epochs = 10
best_val_acc = 0.0
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print("开始训练...")
print("-" * 60)

for epoch in range(num_epochs):
    # 训练
    train_loss, train_acc = train_epoch(model, train_dataloader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # 验证
    val_loss, val_acc = validate_epoch(model, val_dataloader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # 更新学习率
    scheduler.step()
    
    # 打印结果
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'训练 - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%')
    print(f'验证 - Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%')
    print(f'学习率: {scheduler.get_last_lr()[0]:.6f}')
    
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_resnet18_model.pth')
        print(f'✓ 保存最佳模型 (验证准确率: {val_acc:.2f}%)')
    
    print("-" * 60)

print(f'\n训练完成！最佳验证准确率: {best_val_acc:.2f}%')

开始训练...
------------------------------------------------------------


Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=80, pipe_handle=114)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nanako/anaconda3/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/nanako/anaconda3/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'WildfireDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>


BrokenPipeError: [Errno 32] Broken pipe

In [39]:
# 在测试集上评估模型
print("在测试集上评估模型...")
model.load_state_dict(torch.load('best_resnet18_model.pth'))
test_loss, test_acc = validate_epoch(model, test_dataloader, criterion, device)
print(f"测试集 - Loss: {test_loss:.4f}, Acc: {test_acc:.2f}%")

在测试集上评估模型...


FileNotFoundError: [Errno 2] No such file or directory: 'best_resnet18_model.pth'